# Patch-based semantic segmentation

This notebook walks through eotorch's patch-based segmentation workflow:

1. **Create patches**: cut a large raster and a set of labelled polygons into fixed-size image/label patches on disk, with a spatially separate validation set.
2. **Load patches**: wrap the patch folders in a `PatchDataModule`.
3. **Train**: fit a `PatchSegmentationTask` with PyTorch Lightning, and optionally fine-tune it on new data.
4. **Predict**: run sliding-window inference on a full raster and write a class map.

The patch-based workflow suits large, high-resolution rasters (e.g. orthophotos or VRT mosaics) that are only labelled in some areas. Patches are cut only where labels exist, and training reads small pre-cut files instead of windows from the full raster. For the on-the-fly sampling workflow on raster datasets, see `segmentation.ipynb`.

## Setup

The notebook expects:

- A multi-band image raster (GeoTIFF or VRT) at `IMAGE_PATH`.
- One vector file per class in `LABEL_DIR`, named after the class, e.g. `water.shp`, `vegetation.shp`. Any format geopandas reads works, in any CRS; polygons are reprojected to the image CRS.

Classes get the values `1..N` in the order of `CLASSES`. Pixels not covered by any polygon get the value `0` and are ignored by the loss and metrics, so you only need to annotate areas you're sure about. Where polygons of different classes overlap, the later class in `CLASSES` wins.

Patches store the raw band values. Scaling happens on the fly in a transform (section 2), which is reused unchanged at inference:

- `LIMITS` maps each band in `BANDS` from `(low, high)` to `[0, 1]`, keyed by position in `BANDS` (1-based). For 8-bit imagery that is e.g. `(1, 255)`.
- `NODATA` is the raw value of missing pixels. They are set to 0 and left out of the band statistics.

In [ ]:
import json
from pathlib import Path

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import rasterio as rst
import torch
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from eotorch.data import PatchDataModule, PatchSegmentationTask
from eotorch.inference.inference import predict_on_tif_generic
from eotorch.plot import plot_class_raster
from eotorch.processing import clear_patches, generate_train_val_patches
from eotorch.processing.patch import compute_patch_stats
from eotorch.transforms import BandScaling, Compose, Normalize, SelectBands

# Adjust these to your setup
DATA_DIR = Path.home() / "data"
IMAGE_PATH = DATA_DIR / "image.tif"  # multi-band raster or VRT
LABEL_DIR = DATA_DIR / "labels"  # one vector file per class
LABEL_EXT = ".shp"
WORK_DIR = DATA_DIR / "patch_segmentation"

CLASSES = ["water", "vegetation", "built_up"]  # label file stems; class values 1..N
BANDS = [1, 2, 3, 4]  # 1-based band indexes read from IMAGE_PATH
LIMITS = {i: (1, 255) for i in range(1, len(BANDS) + 1)}  # per-band (low, high) scaled to [0, 1]
NODATA = 0  # raw value of missing pixels
PATCH_SIZE = 512

TRAIN_PATCH_DIR = WORK_DIR / "patches" / "train"
VAL_PATCH_DIR = WORK_DIR / "patches" / "val"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
PREDICTION_DIR = WORK_DIR / "predictions"

## 1. Create training and validation patches

`generate_train_val_patches` lays a grid of `PATCH_SIZE` blocks over the raster and keeps the blocks that intersect `feature_shps`:

- A fraction `val_fraction` of the blocks is reserved for validation, with one grid-aligned patch per block.
- Each remaining block yields `n_random_offsets + 1` training patches at random offsets. Patches that would overlap a validation block are skipped, so training and validation pixels never mix.

For each patch, the bands in `indexes` are saved unscaled as `*_image.tiff`. The polygons in `label_sources` are rasterized into `*_label.tiff`, using `class_map` to map each label file's stem to its class value.

Here the label files also decide where patches are cut. To restrict patches to an area of interest, pass a different `feature_shps`, such as an AOI polygon.

In [ ]:
label_files = [LABEL_DIR / f"{c}{LABEL_EXT}" for c in CLASSES]
class_map = {c: i for i, c in enumerate(CLASSES, start=1)}

# Remove patches from a previous run so stale files don't mix with the new set
clear_patches("*", TRAIN_PATCH_DIR)
clear_patches("*", VAL_PATCH_DIR)

generate_train_val_patches(
    src_path=IMAGE_PATH,
    feature_shps=label_files,
    train_patch_dir=TRAIN_PATCH_DIR,
    val_patch_dir=VAL_PATCH_DIR,
    patch_size=PATCH_SIZE,
    val_fraction=0.2,
    n_random_offsets=2,
    indexes=BANDS,
    label_sources=label_files,
    class_map=class_map,
    random_seed=42,
)

**Labels as a raster?** Use `generate_patches_from_files(img_path, label_path, out_dir, patch_size=...)` instead. It writes the same `*_image.tiff` / `*_label.tiff` pairs to `out_dir/train` and `out_dir/val`, and accepts a label raster with a different CRS, resolution or extent than the image. Like here, the image is saved unscaled, so the transforms in section 2 apply unchanged.

### Inspect a few patches

Check that labels line up with the imagery. Images are scaled with `BandScaling` for display, and `rgb` selects the patch bands to show (0-based positions in `BANDS`).

In [ ]:
band_scaling = BandScaling(LIMITS, nodata=NODATA)


def show_patches(patch_dir, n=4, rgb=(0, 1, 2), seed=0):
    image_paths = sorted(Path(patch_dir).glob("*_image.tiff"))
    rng = np.random.default_rng(seed)
    picks = rng.choice(len(image_paths), size=min(n, len(image_paths)), replace=False)

    fig, axes = plt.subplots(2, len(picks), figsize=(4 * len(picks), 8), squeeze=False)
    for col, idx in enumerate(picks):
        img_path = image_paths[idx]
        label_path = img_path.with_name(img_path.name.replace("_image.tiff", "_label.tiff"))
        with rst.open(img_path) as src:
            img = band_scaling(image=src.read())["image"].numpy()[list(rgb)]
        with rst.open(label_path) as src:
            label = src.read(1)

        axes[0, col].imshow(np.moveaxis(img, 0, -1))
        axes[0, col].set_title(img_path.stem)
        axes[1, col].imshow(label, cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
        axes[1, col].set_title("label (0 = unlabelled)")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_patches(TRAIN_PATCH_DIR)

## 2. Input transforms

Inputs are scaled and normalized on the fly, in two steps:

1. `BandScaling` maps each band to `[0, 1]` with `LIMITS` and sets `NODATA` pixels to 0.
2. `Normalize` standardizes each band to zero mean and unit variance. This usually helps training converge, and pretrained backbones such as DINOv3 expect it.

`compute_patch_stats` computes the per-band mean and standard deviation over all patches, excluding nodata pixels. Passing `transform=band_scaling` makes the statistics describe the scaled values that `Normalize` will receive. `Compose` chains both steps into one transform.

Inference must prepare its inputs exactly as training did, so the limits and statistics are saved next to the checkpoints.

**NOTE**: If the values in your image are fairly homogeneous, you can skip `Normalize` and use `band_scaling` alone as the transform.

In [ ]:
mean, std = compute_patch_stats(
    [TRAIN_PATCH_DIR, VAL_PATCH_DIR], image_suffix="image", nodata=NODATA, transform=band_scaling
)
print(f"mean: {mean}")
print(f"std:  {std}")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
transform_path = CHECKPOINT_DIR / "input_transform.json"
transform_path.write_text(json.dumps({
    "limits": [LIMITS[k] for k in sorted(LIMITS)],
    "nodata": NODATA,
    "mean": mean.tolist(),
    "std": std.tolist(),
}))

input_transform = Compose(band_scaling, Normalize(mean, std))

## 3. Data module

`PatchDataModule` pairs `<key>_<image_suffix>.tiff` with `<key>_<label_suffix>.tiff` in each folder. The defaults are `feature` and `label`, so set `image_suffix="image"` for patches made in section 1. If you omit `val_patch_dir`, a random `val_fraction` of the training patches is used for validation instead. That split isn't spatially separate.

`transform` is called per sample as `transform(image=..., mask=...)` and must return a dict with `image` and `mask` keys. To add augmentations (flips, rotations, ...), append them to the `Compose`. They must move the image and mask together.

In [ ]:
BATCH_SIZE = 8

dm = PatchDataModule(
    train_patch_dir=TRAIN_PATCH_DIR,
    val_patch_dir=VAL_PATCH_DIR,
    batch_size=BATCH_SIZE,
    transform=input_transform,
    image_suffix="image",
    label_suffix="label",
    num_workers=4,  # set to 0 to debug data loading in the main process
)
print(f"{len(dm.train_dataset)} training / {len(dm.val_dataset)} validation patches of {dm.patch_size} px")

## 4. Model

`PatchSegmentationTask` holds the model, loss, optimizer and metrics.

- `num_classes` includes the unlabelled class 0, so it is `len(CLASSES) + 1`. `ignore_index=0` excludes those pixels from the loss and metrics.
- `model` is either an eotorch model (`"deepresunet"`, `"unet_lite"`, `"dinov3_upernet"`) or a segmentation-models-pytorch architecture (`"unet"`, `"unetplusplus"`, `"upernet"`, `"segformer"`, `"dpt"`). smp architectures take any smp encoder as `backbone`, and `weights=True` initializes it with ImageNet weights.
- `loss="dice"` uses multiclass Dice loss, which is robust to class imbalance. Other options include `"ce"`, `"focal"`, `"tversky"` and `"jaccard"`.
- The default learning-rate schedule reduces `lr` by a factor of 5 after 10 epochs without improvement in `val_loss`. Pass `lr_scheduler={"type": ..., **kwargs}` to use another `torch.optim.lr_scheduler`.

In [ ]:
class_names = ["unlabelled", *CLASSES]

model = PatchSegmentationTask(
    num_classes=len(class_names),
    in_channels=len(BANDS),
    ignore_index=0,
    model="unetplusplus",
    backbone="resnet34",
    weights=False,
    task="multiclass",
    loss="dice",
    lr=1e-4,
    class_names=class_names,
)

## 5. Train

Validation metrics are logged as `val_loss`, macro-averaged `val/F1_Score`, `val/mIoU`, `val/Accuracy`, and per class `val/IoU_<class>`, `val/Class_F1_<class>` etc.

Each run writes its checkpoints to `CHECKPOINT_DIR / RUN_NAME`:

- `best_f1.ckpt`: best validation F1
- `best_loss.ckpt`: lowest validation loss
- `last.ckpt`: latest epoch

If a run is interrupted, re-running the cell resumes it from `last.ckpt`. Use a new `RUN_NAME` to start a fresh run.

In [ ]:
RUN_NAME = "unet_v1"
MAX_EPOCHS = 100

run_dir = CHECKPOINT_DIR / RUN_NAME

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="auto",
    devices=1,
    logger=CSVLogger(WORK_DIR / "logs", name=RUN_NAME),  # or e.g. WandbLogger(project=..., name=RUN_NAME)
    callbacks=[
        LearningRateMonitor(logging_interval="epoch"),
        ModelCheckpoint(dirpath=run_dir, filename="best_f1", monitor="val/F1_Score", mode="max", save_last=True),
        ModelCheckpoint(dirpath=run_dir, filename="best_loss", monitor="val_loss", mode="min"),
    ],
)

trainer.fit(model, datamodule=dm, ckpt_path="last")  # starts from scratch if there is no last.ckpt

### Fine-tune on new data (optional)

To adapt a trained model to new data, such as imagery from another date or newly annotated areas:

1. Create patches for the new data as in section 1.
2. Continue training from a checkpoint with a lower learning rate. `load_from_checkpoint` restores the architecture and weights, and its keyword arguments override the saved hyperparameters.

Reuse `input_transform` from the original training, so the new inputs are scaled and normalized the same way as the data the model was trained on.

In [ ]:
FINETUNE_RUN_NAME = f"{RUN_NAME}_finetuned"
FINETUNE_TRAIN_DIR = WORK_DIR / "patches_finetune" / "train"
FINETUNE_VAL_DIR = WORK_DIR / "patches_finetune" / "val"

finetune_dm = PatchDataModule(
    train_patch_dir=FINETUNE_TRAIN_DIR,
    val_patch_dir=FINETUNE_VAL_DIR,
    batch_size=BATCH_SIZE,
    transform=input_transform,
    image_suffix="image",
    label_suffix="label",
    num_workers=4,
)

finetune_model = PatchSegmentationTask.load_from_checkpoint(run_dir / "best_f1.ckpt", lr=1e-5)

finetune_dir = CHECKPOINT_DIR / FINETUNE_RUN_NAME
finetune_trainer = L.Trainer(
    max_epochs=50,
    accelerator="auto",
    devices=1,
    logger=CSVLogger(WORK_DIR / "logs", name=FINETUNE_RUN_NAME),
    callbacks=[
        LearningRateMonitor(logging_interval="epoch"),
        ModelCheckpoint(dirpath=finetune_dir, filename="best_f1", monitor="val/F1_Score", mode="max", save_last=True),
        ModelCheckpoint(dirpath=finetune_dir, filename="best_loss", monitor="val_loss", mode="min"),
    ],
)

finetune_trainer.fit(finetune_model, datamodule=finetune_dm, ckpt_path="last")

## 6. Predict on a full raster

Prediction runs directly on the raw raster. The `transforms` argument prepares each batch the way training did:

1. `SelectBands` keeps the bands in `BANDS`.
2. The input transform, rebuilt from the saved limits and statistics, scales and normalizes them.

`predict_on_tif_generic` slides a `PATCH_SIZE` window over the raster with 50 % overlap (`overlap=2`) and writes the per-class probabilities to a GeoTIFF. With `blend=True`, overlapping windows are merged with a weighted average that favours each window's centre, which avoids visible seams between patches.

Blending holds the whole output in memory (`num_classes` float32 bands). For very large rasters, either split the raster into tiles first, or pass `prediction_func=model.predict_class` with `blend=False` to write class indices directly.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PatchSegmentationTask.load_from_checkpoint(run_dir / "best_f1.ckpt", map_location=device)

params = json.loads(transform_path.read_text())
input_transform = Compose(
    BandScaling(params["limits"], nodata=params["nodata"]),
    Normalize(params["mean"], params["std"]),
)

PREDICT_IMAGE_PATH = IMAGE_PATH  # any raster with the same bands as the training image
stem = Path(PREDICT_IMAGE_PATH).stem

prob_path = PREDICTION_DIR / f"{stem}_{RUN_NAME}_prob.tif"
predict_on_tif_generic(
    tif_file_path=PREDICT_IMAGE_PATH,
    prediction_func=model.predict_probability,
    patch_size=PATCH_SIZE,
    batch_size=4,
    overlap=2,
    transforms=[SelectBands(BANDS), input_transform],
    blend=True,
    dtype="float32",
    num_bands=model.hparams["num_classes"],
    compress="deflate",
    out_file_path=prob_path,
)

**Faster inference on GPU:** wrap the prediction function in mixed precision. Softmax runs in float32 under autocast, so the probabilities stay float32:

```python
def predict_probability_amp(batch):
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        return model.predict_probability(batch)
```

Then pass `prediction_func=predict_probability_amp`.

### Class map

Take the most likely class per pixel. Channel 0 belongs to the ignored unlabelled class and never received a training signal, so it is excluded from the argmax. Pixels that are nodata in the input are set to `0`.

In [ ]:
with rst.open(prob_path) as src:
    probs = src.read()
    profile = src.profile
with rst.open(PREDICT_IMAGE_PATH) as src:
    valid = (src.read(BANDS) != NODATA).any(axis=0)

class_map_arr = (probs[1:].argmax(axis=0) + 1).astype("uint8")
class_map_arr[~valid] = 0

class_path = PREDICTION_DIR / f"{stem}_{RUN_NAME}_classes.tif"
profile.update(count=1, dtype="uint8", nodata=0, compress="deflate")
profile.pop("predictor", None)
with rst.open(class_path, "w", **profile) as dst:
    dst.write(class_map_arr, 1)

fig, ax = plt.subplots(figsize=(10, 10))
plot_class_raster(class_path, ax=ax, class_mapping={i: c for i, c in enumerate(CLASSES, start=1)})
plt.show()